<a href="https://colab.research.google.com/github/Geraldand/2026SpringStat2-Final-Individual-Project/blob/main/notebooks%20/%20Final_Individual_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Data cleaning and value explanned

In [5]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# 1. Load the original dataset
# Ensure 'YRBS_2007 (1).csv' is in the same directory as your script/notebook
df = pd.read_csv('YRBS_2007 (1).csv')

# 2. Define target variables for the research question
# Independent Variable (X)
x_var = 'SafetyConcernsAtSchool'

# Components for the Dependent Variable (Y)
y_components = [
    'WeaponCarrying',
    'GunCarryingPast12Mos',
    'WeaponCarryingAtSchool',
    'PhysicalFighting'
]

# Control / Deep Insight Variables
control_vars = ['WhatIsYourSex', 'SadOrHopeless']

# Combine all target variables for initial filtering
all_target_vars = [x_var] + y_components + control_vars

# 3. Handle Missing Data
# Drop rows with missing values (NaN) within our targeted variables
df_clean = df[all_target_vars].dropna().copy()

# Convert components to numeric, coercing any invalid characters into NaN, then drop them
for col in [x_var] + y_components:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
df_clean = df_clean.dropna()

# 4. Recode Independent Variable (X): Convert categories into actual days
# Original Code Mapping from User Guide:
# 1 = 0 days, 2 = 1 day, 3 = 2 or 3 days, 4 = 4 or 5 days, 5 = 6 or more days
safety_mapping = {
    1: 0.0,
    2: 1.0,
    3: 2.5,
    4: 4.5,
    5: 6.0
}
df_clean['Safety_Days_Missed'] = df_clean[x_var].map(safety_mapping)

# 5. Recode and Aggregate Dependent Variable (Y): Create 'Violence_Risk_Score'
# In YRBS, 1 usually means "0 times / No", and values > 1 mean "Yes (with different frequencies)".
# We binarize these into: Yes = 1, No = 0
for col in y_components:
    df_clean[f'{col}_bin'] = np.where(df_clean[col] > 1, 1, 0)

# Sum the 4 binary behaviors to create a continuous score ranging from 0 to 4
df_clean['Violence_Risk_Score'] = (
    df_clean['WeaponCarrying_bin'] +
    df_clean['GunCarryingPast12Mos_bin'] +
    df_clean['WeaponCarryingAtSchool_bin'] +
    df_clean['PhysicalFighting_bin']
)

# 6. Recode Control Variables (Dummy Encoding)
# Gender: Original code (1 = Female, 2 = Male) -> Convert to Is_Male (1 = Male, 0 = Female)
df_clean['Is_Male'] = np.where(df_clean['WhatIsYourSex'] == 2, 1, 0)

# Depression: Original code (1 = Yes, 2 = No) -> Convert to Is_Sad (1 = Sad/Hopeless, 0 = No)
df_clean['Is_Sad'] = np.where(df_clean['SadOrHopeless'] == 1, 1, 0)

# 7. Filter and Export the Cleaned Dataset
final_features = ['Safety_Days_Missed', 'Violence_Risk_Score', 'Is_Male', 'Is_Sad']
cleaned_df = df_clean[final_features].dropna()

# Save to CSV for project repository compliance
cleaned_df.to_csv('cleaned_data.csv', index=False)
print("✅ Data cleaning complete!")
print(f"Original records: {len(df)} | Cleaned valid records: {len(cleaned_df)}")
print("File successfully saved as 'cleaned_data.csv'\n")

# 8. Run Multiple Linear Regression
# Define predictors (X) and append a constant term for the intercept
X = cleaned_df[['Safety_Days_Missed', 'Is_Male', 'Is_Sad']]
X = sm.add_constant(X)
Y = cleaned_df['Violence_Risk_Score']

# Fit the OLS model
model = sm.OLS(Y, X).fit()

# Display the statistical summary report
print("             ====== Multiple Linear Regression Model Summary ======")
print(model.summary())

✅ Data cleaning complete!
Original records: 14041 | Cleaned valid records: 12997
File successfully saved as 'cleaned_data.csv'

             ====== Multiple Linear Regression Model Summary ======
                             OLS Regression Results                            
Dep. Variable:     Violence_Risk_Score   R-squared:                       0.130
Model:                             OLS   Adj. R-squared:                  0.130
Method:                  Least Squares   F-statistic:                     649.2
Date:                 Thu, 18 Jun 2026   Prob (F-statistic):               0.00
Time:                         14:51:37   Log-Likelihood:                -16899.
No. Observations:                12997   AIC:                         3.381e+04
Df Residuals:                    12993   BIC:                         3.384e+04
Df Model:                            3                                         
Covariance Type:             nonrobust                                         
    



---

## Part 1: Data Preparation & Cleaning

### 1. Feature Selection and Filtering

* **What was done:** The script filters the massive `YRBS_2007 (1).csv` dataset to retain only 7 specific variables required for this research question: `SafetyConcernsAtSchool`, `WeaponCarrying`, `GunCarryingPast12Mos`, `WeaponCarryingAtSchool`, `PhysicalFighting`, `WhatIsYourSex`, and `SadOrHopeless`. Rows containing missing values (`NaN`) or non-numeric entries were strictly dropped.
* **Why it was done:** The original dataset contains hundreds of variables, most of which act as statistical noise for our specific inquiry. By subsetting and dropping incomplete survey responses, we eliminate data corruption, satisfy the project's requirement for rigorous "missing data handling," and optimize computational efficiency.

### 2. Recoding the Independent Variable (X) to Continuous Scale

* **What was done:** The categorical survey codes for school truancy due to safety fears (`SafetyConcernsAtSchool`) were mapped to mathematical averages of actual days: `1` $\rightarrow$ `0.0` days, `2` $\rightarrow$ `1.0` day, `3` $\rightarrow$ `2.5` days, `4` $\rightarrow$ `4.5` days, and `5` $\rightarrow$ `6.0` days.
* **Why it was done:** In the raw dataset, the numbers 1 to 5 are merely nominal codes representing descriptive intervals (e.g., code 3 means "2 or 3 days"). Treating ordinal codes as continuous metrics is a major statistical flaw. By converting them into estimated actual days, we successfully transform X into a truly continuous numeric scale suited for Linear Regression.

### 3. Aggregating the Dependent Variable (Y): `Violence_Risk_Score`

* **What was done:** The four weapon and fighting variables were binarized using `np.where`. If a student engaged in the behavior (survey code > 1), they received `1` point; if they did not (survey code == 1), they received `0` points. These four indicators were then summed up into a new composite index named `Violence_Risk_Score` (ranging from 0 to 4).
* **Why it was done:** Individually, these 4 behaviors are binary "Yes/No" outcomes, which violate the mathematical assumptions of Standard Linear Regression (which requires a continuous response variable). Aggregating them into a scale quantifies a student's cumulative behavioral intensity. This maximizes the analytical depth by highlighting the compounding nuances of trauma-induced defensive behaviors.

### 4. Dummy Encoding Control Variables

* **What was done:** `WhatIsYourSex` (where 2 represented Male) was mapped into a new column `Is_Male` (1 = Male, 0 = Female). Similarly, `SadOrHopeless` (where 1 represented Yes) was mapped to `Is_Sad` (1 = Experiencing severe sadness, 0 = No).
* **Why it was done:** Regression models cannot process raw categorical text or arbitrary nominal scales. Converting them into binary indicators (0 or 1) allows the OLS algorithm to evaluate their impact. Including them isolates confounding factors, enabling us to measure the true, uncorrupted effect of school fear on violence.

---

## Part 2: Numerical & Variable Summary

The finalized file `cleaned_data.csv` contains exactly **4 clean columns** representing the consolidated dataset for the regression model.

### 1. Variable Descriptions

* **`Safety_Days_Missed` (Independent Variable / X):** Continuous scale representing the estimated number of days a student skipped school in the past 30 days due to safety concerns (Values: 0.0, 1.0, 2.5, 4.5, 6.0).
* **`Violence_Risk_Score` (Dependent Variable / Y):** An additive composite metric representing a student's total engagement in violence and weapon possession over the tracking period (Discrete Continuous approximation; Values: 0 to 4).
* **`Is_Male` (Control Variable / Background):** A binary indicator capturing biological sex. Controlled to prevent gender imbalances from skewing environmental effect parameters (Values: 1 for Male, 0 for Female).
* **`Is_Sad` (Control Variable / Mental Health):** A binary indicator capturing prolonged depressive affect. Controlled to isolate the overlapping behavioral changes brought on by internal psychological trauma (Values: 1 for Sad/Hopeless, 0 for Normal).

### 2. Generated Outputs from the Script

* **`cleaned_data.csv`:** A streamlined table containing only the 4 curated metrics above, stripped of all hundreds of original redundant columns.
* **OLS Regression Table:** The final execution stage outputs an Ordinary Least Squares (OLS) summary containing key statistical parameter arrays (`coefficients` to show effect directions, `p-values` to determine significance, and `R-squared` to show total model variance explanation).

---

## 第一部分：資料準備與清理邏輯

### 1. 特徵選擇與樣本篩選 (Feature Selection & Filtering)

* **做了什麼：** 程式碼從龐大的 `YRBS_2007 (1).csv` 原始資料中，精準抽取出與本研究主題相關的 7 個變數：`SafetyConcernsAtSchool`、`WeaponCarrying`、`GunCarryingPast12Mos`、`WeaponCarryingAtSchool`、`PhysicalFighting`、`WhatIsYourSex` 以及 `SadOrHopeless`。接著，程式會自動將任何包含空白、缺失值（NaN）或無效字元的整行學生數據徹底刪除。
* **為什麼這樣做：** 原始資料集包含數百個不相關的問卷問題，對本專案而言屬於統計雜訊。透過過濾並排除填答不全的樣本，不僅能防止程式運行報錯，更能有效落實專案評分標準中強調的「遺漏值處理（Missing Data Handling）」，大幅提升數據模型的乾淨度。

### 2. 自變數 (X) 轉化為連續型天數 (Recoding X)

* **做了什麼：** 將原始問卷中代表「因安全疑慮不敢上學」的類別代號（1~5），映射轉化為代表實際天數的平均數值：代號 `1` $\rightarrow$ `0.0` 天、`2` $\rightarrow$ `1.0` 天、`3` $\rightarrow$ `2.5` 天、`4` $\rightarrow$ `4.5` 天、`5` $\rightarrow$ `6.0` 天。
* **為什麼這樣做：** 原始數據中的 1 到 5 只是代表某個區間的「類別代碼」（例如代號 3 代表 2 或 3 天）。在統計學中，直接將類別代碼當作連續數字跑線性迴歸是嚴重的邏輯錯誤。將其轉換為估算天數後，自變數 X 才能變成真正的**連續變數**，完全符合線性迴歸模型的數學假設。

### 3. 衍生建立依變數 (Y)：「暴力與武器風險指數 (`Violence_Risk_Score`)」

* **做了什麼：** 運用 `np.where` 將四項關於武器與打架的行為指標進行「二值化（Binarize）」處理。若學生在過去一段時間曾涉入該行為（原始代碼 > 1）則計為 `1` 分；若無（原始代碼 == 1）則計為 `0` 分。最後，將這四個項目的得分加總，融合成一個範圍在 0 到 4 分之間的新指標，命名為 `Violence_Risk_Score`。
* **為什麼這樣做：** 單一的暴力或攜帶武器行為在問卷中通常只是「有」或「沒有」的二分法，這不符合標準線性迴歸必須使用連續型依變數的要求。將多個高風險行為進行權重加總，能將學生的行為表現轉化為具備「程度連續性」的風險指數。這樣做不僅能完美契合迴歸模型的計算，更能放大觀察學生在恐懼狀態下的細微行為質變，大幅拉高研究深度。

### 4. 控制變數的虛擬化編碼 (Dummy Encoding)

* **做了什麼：** 將性別欄位 `WhatIsYourSex`（原始代碼 2 代表男性）轉化為新欄位 `Is_Male`（1 = 男生，0 = 女生）；將憂鬱狀態 `SadOrHopeless`（原始代碼 1 代表有絕望感）轉化為新欄位 `Is_Sad`（1 = 過去一年曾連續兩週感到悲傷絕望，0 = 無）。
* **為什麼這樣做：** 迴歸模型無法直接判讀文字或無意義的分類代碼。透過將其轉化為只有 0 與 1 的「虛擬變數（Dummy Variable）」，統計模型才能正確運算。引入這兩個變數作為控制變數，能排除性別天生差異與內在心理創傷的干擾，讓我們看清校園不安全感對暴力行為的「純粹環境影響力」。

---

## 第二部分：數值與變數總整理

經過上述清理後，最終產出的 `cleaned_data.csv` 檔案中**僅保留了 4 個欄位**，這正是統計模型所需的精華資料。

### 1. 變數說明 (Variable Descriptions)

* **`Safety_Days_Missed`（自變數 / Independent Variable X）：** 連續變數，代表學生在過去 30 天內，因為覺得學校或上下學路上不安全而不敢上學的估算天數（數值範圍：0.0, 1.0, 2.5, 4.5, 6.0）。
* **`Violence_Risk_Score`（依變數 / Dependent Variable Y）：** 連續變數（趨近），代表學生在暴力行為與武器攜帶上的綜合涉入程度。分數越高，代表行為越激進或防禦性武裝越嚴重（數值範圍：0 至 4 分的整數）。
* **`Is_Male`（控制變數 / 學生背景）：** 二元類別變數，記錄學生的生理性別。加入此變數可避免模型因男女樣本比例不均而產生評估偏差（1 = 男生，0 = 女生）。
* **`Is_Sad`（控制變數 / 心理健康）：** 二元類別變數，記錄學生長期的憂鬱絕望感。用於控制學生內在心理創傷所帶來的行為改變細節（1 = 有悲傷絕望感，0 = 無）。

### 2. 程式碼執行後產生的最終成品

* **`cleaned_data.csv`：** 這是一張剔除了原始數百個無用欄位、只保留上述 4 個核心精華指標的乾淨資料表。
* **OLS 迴歸分析報表 (OLS Regression Summary)：** 程式碼的最後階段會直接在 Jupyter Notebook 螢幕上印出完整的統計報表。這份報表包含了核心的**斜率 (Coefficients)**、**顯著性檢定 (P-values)** 以及**模型解釋力 (R-squared)**。